In [1]:
!pip install jsonlines

Looking in indexes: https://mirrors.aliyun.com/pypi/simple/
DEPRECATION: pytorch-lightning 1.7.7 has a non-standard dependency specifier torch>=1.9.*. pip 24.0 will enforce this behaviour change. A possible replacement is to upgrade to a newer version of pytorch-lightning or contact the author to suggest that they release a version with a conforming dependency specifiers. Discussion can be found at https://github.com/pypa/pip/issues/12063

[notice] A new release of pip is available: 23.3.2 -> 25.3
[notice] To update, run: pip install --upgrade pip


In [1]:
import os
from typing import List
import jsonlines
from tqdm import tqdm
import random
import torch
import torch.nn as nn
import torch.optim as optim
import json
from torch.utils.data import Dataset, DataLoader

/usr/local/lib/python3.11/site-packages/torch/cuda/__init__.py:56: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]


In [9]:
def load_json(json_file):
    with open(json_file, "r", encoding="utf-8") as reader:
        return json.load(reader)

def save_json(json_file, json_obj):
    with open(json_file, "w", encoding="utf-8") as writer:
        json.dump(json_obj, writer, indent=2, ensure_ascii=False)

def tensor_to_device(batch, device):
    for k, v in batch.items():
        if isinstance(v, torch.Tensor):
            v = v.to(device=device)
            batch[k] = v

@torch.no_grad()
def accuracy(score, target_ids):
    pred_ids = torch.argmax(score, dim=2) # [bs,t,c] -> [bs,t]
    is_equals = pred_ids == target_ids # [bs,t]
    is_target = target_ids != -100 # [bs,t]
    
    acc = torch.sum(is_equals.to(torch.float32)) / torch.sum(is_target.to(torch.float32))
    return acc

___整体思路是：预测每个token属于BMES四个类别中的那一个类别？___
- B表示当前字是一个词的开头；
- M表示当前字是一个词的中间；
- E表示当前字是一个词的结尾；
- S表示当前字独立成词；

```text
原始文本:  我是中国人
对应的实际标签/期望预测标签：SSBME

我 --> S
是 --> S
中 --> B
国 --> M
人 --> E
```

# 一、数据处理

原始数据格式为：
```
19980131-04-013-020/m  一/m  颗/q  被/p  抚慰/v  得/u  微醺/v  的/u  心/n  ，/w  
```
转换后的数据格式:
```json
{"text":"一颗被抚慰得微醺的心,", "label":"SSSBESBESSS"}
```

In [7]:
in_dirs = "./datas/original/"
out_files = "./datas/all_data.jsonl"

# 输出文件夹创建
os.makedirs(os.path.dirname(out_files), exist_ok=True)

# 获取总的原始数据文件
in_names = os.listdir(in_dirs)

# 遍历每个文件进行读取处理
all_datas = []
for in_name in in_names:
    in_file = os.path.join(in_dirs, in_name)
    if not os.path.isfile(in_file):
        continue
    with open(in_file, "r", encoding="utf-8") as reader:
        for line in tqdm(reader):

            # 解析当前行原始数据
            text = []
            label = [] 
            line = line.strip()
            if line:
                tokens = line.split(" ")[1:]
                for token in tokens:
                    token = token.strip()
                    if token:
                        try:
                            word, _ = token.split("/")
                            word_len = len(word)
                            if word_len == 1:
                                label.append("S")
                            elif word_len == 2:
                                label.append("BE")
                            else:
                                label.append("B")
                                label.append("M" * (word_len - 2))
                                label.append("E")
                            text.append(word)
                        except Exception as e:
                            print(f"解析异常 {line} {token} {e}")
                text = ''.join(text)
                label = ''.join(label)
                assert len(text) == len(label), f"代码异常，text和label数量不匹配: {text} {label}"
                all_datas.append({'text': text, 'label': label})

# 所有数据输出
with jsonlines.open(out_files, "w") as writer:
    writer.write_all(all_datas)
print(f"输出所有数据完成:{out_files} - {len(all_datas)}")

23064it [00:00, 25977.10it/s]


输出所有数据完成:./datas/all_data.jsonl - 19484


In [8]:
all_data_file = "./datas/all_data.jsonl"
train_data_file = "./datas/train.jsonl"
test_data_file = "./datas/test.jsonl"

# 输出文件夹创建
os.makedirs(os.path.dirname(train_data_file), exist_ok=True)
os.makedirs(os.path.dirname(test_data_file), exist_ok=True)

# 加载所有数据, 并按照随机分配给训练集和测试集
train_datas, test_datas = [], []
test_size = 0.2
with jsonlines.open(all_data_file, "r") as reader:
    for obj in reader:
        if random.random() < test_size:
            test_datas.append(obj)
        else:
            train_datas.append(obj)

# 数据输出
with jsonlines.open(train_data_file, "w") as writer:
    writer.write_all(train_datas)
print(f"输出训练数据完成:{train_data_file} - {len(train_datas)}")
with jsonlines.open(test_data_file, "w") as writer:
    writer.write_all(test_datas)
print(f"输出测试数据完成:{test_data_file} - {len(test_datas)}")

输出训练数据完成:./datas/train.jsonl - 15556
输出测试数据完成:./datas/test.jsonl - 3928


# 二、模型训练

模型整体结构：分字 --> 字转Token id ---> Embedding获取Token向量 ---> 通过多层、双向LSTM结构提取/更新Token向量 --> 针对每个token向量进行全连接操作获取每个token属于BMES四个类别的置信度；

#### __2.1 构建网络结构__
输入数据X: 输入bs个样本，每个样本t个token，shape形状为：\[bs,t], 取值类型：long；

实际数据标签Y: 针对每个token对应一个类别标签id，shape形状为: \[bs,t], 取值类型：long;

模型前行输出结果：针对每个token预测属于每个类别对应的置信度值，shape形状为: \[bs,t,c]，取值类型：float；

** __bs个样本中，每个样本的实际token数目是不一样的，所以输入存在数据填充(X和Y)__

In [10]:
class TokenClassifyNetwork(nn.Module):
    def __init__(self, vocab_size, num_classes, hidden_size, num_layers=3):
        super().__init__()
        self.embed_layer = nn.Embedding(num_embeddings=vocab_size, embedding_dim=hidden_size)
        self.rnn_layers = nn.LSTM(
            input_size=hidden_size, hidden_size=hidden_size,
            num_layers=num_layers, bias=True, batch_first=True,
            bidirectional=False
        )
        self.classify = nn.Sequential(
            nn.Linear(hidden_size, 2 * hidden_size),
            nn.ReLU(),
            nn.Dropout(p=0.3),
            nn.Linear(2*hidden_size, num_classes)
        )
        

    def forward(self, input_token_ids):
        """
        前行执行过程
            bs: 样本数目，也就是批次大小
            t: 每个样本的token数目，也就是样本序列长度
            e: 用来表示网络中向量维度大小  不同阶段的时候，该符号表示的实际数值可能会变化
            c: 类别数目
        """
        # 1. 获取输入token对应的特征向量: 相当于哑编码 --> 矩阵乘法
        # [bs,t] -> [bs,t,e]
        token_emb = self.embed_layer(input_token_ids)

        # 2. 通过LSTM提取特征
        # rnn_output: RNN/LSTM/GRU网络的每个token对应输出特征向量 [bs,t,e]
        rnn_output, _ = self.rnn_layers(token_emb)

        # 3. 针对每个token进行分类判断决策
        # [bs,t,e] * [e,c] -> [bs,t,c]
        score = self.classify(rnn_output)

        return score

In [10]:
# 测试案例
network = TokenClassifyNetwork(vocab_size=100, num_classes=4, hidden_size=128)
print(f"网络结构\n{network}\n")
input_token_ids = torch.randint(0, 100, size=(2,5))
output_scores = network(input_token_ids)
print(f"前向输入:\n{input_token_ids}\n")
print(f"前向输出:\n{output_scores}\n")


网络结构
TokenClassifyNetwork(
  (embed_layer): Embedding(100, 128)
  (rnn_layers): LSTM(128, 128, num_layers=3, batch_first=True)
  (classify): Sequential(
    (0): Linear(in_features=128, out_features=256, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.3, inplace=False)
    (3): Linear(in_features=256, out_features=4, bias=True)
  )
)

前向输入:
tensor([[98, 59, 19, 52, 45],
        [41, 80, 43, 13, 71]])

前向输出:
tensor([[[-0.0812,  0.0753, -0.0131,  0.0420],
         [-0.0502,  0.0760, -0.0294,  0.0616],
         [-0.0903,  0.0527, -0.0182,  0.0558],
         [-0.0633,  0.0457, -0.0210,  0.0553],
         [-0.0330,  0.0713, -0.0191,  0.0778]],

        [[-0.0812,  0.0732, -0.0015,  0.0345],
         [-0.0705,  0.0745,  0.0031,  0.0335],
         [-0.0558,  0.0511, -0.0321,  0.0550],
         [-0.0988,  0.0651, -0.0337,  0.0495],
         [-0.0716,  0.0648, -0.0026,  0.0689]]], grad_fn=<ViewBackward0>)



#### __2.2 损失计算__
输入数据X: 输入bs个样本，每个样本t个token，shape形状为：\[bs,t], 取值类型：long；

实际数据标签Y: 针对每个token对应一个类别标签id，shape形状为: \[bs,t], 取值类型：long;

模型前行输出结果：针对每个token预测属于每个类别对应的置信度值，shape形状为: \[bs,t,c]，取值类型：float；

** __bs个样本中，每个样本的实际token数目是不一样的，所以输入存在数据填充(X和Y)；针对填充位置的预测，不需要计算损失（实际标签数据，填充位置使用-100来填充）__

In [11]:
class TokenClassifyLossModule(nn.Module):
    def __init__(self, reduction='mean'):
        super().__init__()
        # 实际标签为-100的位置不计算损失(损失为0)
        self.loss_fn = nn.CrossEntropyLoss(ignore_index=-100, reduction=reduction)

    def forward(self, score, targets):
        """
            计算损失
        """
        score = torch.permute(score, dims=(0,2,1))
        return self.loss_fn(score, targets)

In [12]:
# 测试案例
input_targets = torch.tensor([
    [3,3,0,1,2], # SSBME
    [3,3,-100,-100,-100] # SSS
])
print(f"实际标签(填充后):\n{input_targets}\n")
bs,t = input_targets.shape
output_scores = torch.rand(bs,t,4)
print(f"前向输出:\n{output_scores}\n")

loss_fn1 = TokenClassifyLossModule(reduction='none')
loss1 = loss_fn1(output_scores, input_targets)
print(f"损失值为:\n{loss1}\n")

loss_fn2 = TokenClassifyLossModule()
loss2 = loss_fn2(output_scores, input_targets)
print(f"损失值为:\n{loss2}\n")


实际标签(填充后):
tensor([[   3,    3,    0,    1,    2],
        [   3,    3, -100, -100, -100]])

前向输出:
tensor([[[0.8827, 0.8290, 0.1777, 0.8124],
         [0.7881, 0.2027, 0.5259, 0.0436],
         [0.1569, 0.5404, 0.5888, 0.3686],
         [0.7765, 0.1975, 0.2706, 0.8779],
         [0.3413, 0.4705, 0.8448, 0.9087]],

        [[0.2211, 0.0568, 0.3294, 0.2477],
         [0.8393, 0.6671, 0.4721, 0.9374],
         [0.0432, 0.2120, 0.0288, 0.5931],
         [0.7031, 0.8201, 0.9686, 0.6474],
         [0.4976, 0.1122, 0.9897, 0.8768]]])

损失值为:
tensor([[1.2864, 1.7745, 1.6570, 1.7638, 1.2114],
        [1.3572, 1.1932, 0.0000, 0.0000, 0.0000]])

损失值为:
1.4633606672286987



#### __2.3 数据集加载__

构造数据加载的自定义Dataset对象，并将原始的token转换为token id，原始的标签转换为标签id

1. 构造token到token id的映射mapping，内部包含两个两个特殊token：PAD、UNK, 针对整个数据中出现次数少于10次的token直接删除(不构建该token的映射，直接当成未知词)；
2. 构建类别标签名称到标签id的映射mapping；
3. 构建自定义Dataset以及DataLoader

__PS：需要考虑填充__

In [13]:
## 统计每个token出现的次数，看看哪些是出现次数少的token

data_files = "./datas/all_data.jsonl"
few_token_files = "./datas/tmp/few_tokens.json"
token2cnt = {}

os.makedirs(os.path.dirname(few_token_files), exist_ok=True)

# 加载所有数据, 并按照随机分配给训练集和测试集
with jsonlines.open(data_files, "r") as reader:
    for obj in reader:
        text = obj['text']
        for token in list(text):
            token = token.lower() # 全部转换为小写
            token2cnt[token] = token2cnt.get(token, 0) + 1

few_tokens = []
for token, cnt in token2cnt.items():
    if cnt < 10:
        few_tokens.append([token, cnt])
print(f"出现次数过少的token:{len(few_tokens)} \n{few_tokens[:10]}\n")

# 出现次数少的token
with open(few_token_files, "w", encoding="utf-8") as writer:
    json.dump(few_tokens, writer, indent=2, ensure_ascii=False)

出现次数过少的token:1681 
[['礴', 8], ['袂', 9], ['湟', 3], ['燮', 7], ['闫', 4], ['戌', 3], ['溺', 4], ['锲', 6], ['诛', 2], ['璐', 9]]



In [14]:
# 构建token到id的映射mapping
tokens = ['<PAD>', '<UNK>']

token_json_file = "./datas/tokens.json"

# 创建输出文件夹
os.makedirs(os.path.dirname(token_json_file), exist_ok=True)

# 加载所有数据, 并按照随机分配给训练集和测试集
for token, cnt in token2cnt.items():
    if cnt >= 10:
        tokens.append(token)
print(f"总有效token数目:{len(tokens)}")

# token映射mapping输出
with open(token_json_file, "w", encoding="utf-8") as writer:
    json.dump(tokens, writer, indent=2, ensure_ascii=False)

总有效token数目:2986


In [15]:
# 构建标签映射mapping
labels = ['B', 'M', 'E', 'S']
label_json_files = "./datas/labels.json"
os.makedirs(os.path.dirname(label_json_files), exist_ok=True)

print(f"总标签数目:{len(labels)}")

# token映射mapping输出
with open(label_json_files, "w", encoding="utf-8") as writer:
    json.dump(labels, writer, indent=2, ensure_ascii=False)

总标签数目:4


##### 自定义分词器

In [12]:
class Tokenizer:
    def __init__(self, vocab_path, do_lower_case=True, unk_token='<UNK>', pad_token='<PAD>'):
        super().__init__()

        # 单词的恢复
        self.vocabs = load_json(vocab_path)
        self.id2token_mapping = {k:v for k,v in enumerate(self.vocabs)}
        self.token2id_mapping = {v:k for k,v in enumerate(self.vocabs)}
        self.do_lower_case = do_lower_case

        # 构建特殊token
        self.unk_token = unk_token
        self.unk_token_id = self.token2id_mapping[self.unk_token]
        self.pad_token = pad_token
        self.pad_token_id = self.token2id_mapping[self.pad_token]

    def tokenize(self, text:str) -> List[str]:
        """
            将文本转换为token列表
        """
        if self.do_lower_case:
            text = text.lower()
        split_tokens = list(text)
        return split_tokens
        
    def convert_tokens_to_ids(self, tokens: List[str]) -> List[int]:
        """
            将每个对应的token字符串转换为对应的id
        """
        return [self.convert_token_to_id(token) for token in tokens]

    def convert_token_to_id(self, token:str) -> int:
        return self.token2id_mapping.get(token, self.unk_token_id)

##### 自定义数据加载器

In [13]:
class TokenClassifyDataset(Dataset):
    def __init__(self, jsonl_file, tokenizer:Tokenizer, label2idx, pad_label_idx=-100):
        super().__init__()

        self.tokenizer = tokenizer
        self.label2idx = label2idx
        self.pad_label_idx = pad_label_idx

        # 加载jsonl格式数据
        datas = []
        with jsonlines.open(jsonl_file, "r") as reader:
            for obj in reader:
                token_ids, label_ids = [], []
                for token, label in zip(obj['text'], obj['label']):
                    token = token.lower()
                    token_ids.append(self.tokenizer.convert_token_to_id(token))
                    label_ids.append(self.label2idx[label])
                datas.append({
                    'tokens': obj['text'],
                    'label': obj['label'],
                    'token_ids': token_ids,
                    'label_ids': label_ids
                })
        self.datas = datas

    def __len__(self):
        return len(self.datas)

    def __getitem__(self, index):
        data = self.datas[index]
        return {
            'tokens': data['tokens'],
            'labels': data['label'],
            'token_ids': torch.tensor(data['token_ids'], dtype=torch.long),
            'label_ids': torch.tensor(data['label_ids'], dtype=torch.long),
            'token_len': len(data['token_ids'])
        }

    def build_collate_fn(self):

        _pad_token_idx = self.tokenizer.pad_token_id
        _pad_label_idx = self.pad_label_idx
        def _collate_fn(batch):
            max_token_len = max([item['token_len'] for item in batch])
            token_ids_list,label_ids_list = [], []
            tokens_list, labels_lie = [], []
            for item in batch:
                token_len = item['token_len']
                token_ids = item['token_ids']
                label_ids = item['label_ids']
                if token_len < max_token_len:
                    append_size = max_token_len - token_len
                    token_ids = torch.cat([token_ids, torch.ones(append_size, dtype=torch.long) * _pad_token_idx], dim=0)
                    label_ids = torch.cat([label_ids, torch.ones(append_size, dtype=torch.long) * _pad_label_idx], dim=0)
                token_ids_list.append(token_ids)
                label_ids_list.append(label_ids)
                tokens_list.append(item['tokens'])
                labels_lie.append(item['labels'])
            return {
                'tokens': tokens_list,
                'labels': labels_lie,
                'token_ids': torch.stack(token_ids_list, dim=0),
                'label_ids': torch.stack(label_ids_list, dim=0)
            }
        return _collate_fn


In [18]:
# 数据代码校验测试
tokenizer = Tokenizer("./datas/tokens.json")
label2idx = {k:v for v, k in enumerate(labels)}
ds = TokenClassifyDataset(
    jsonl_file=train_data_file,
    tokenizer=tokenizer, 
    label2idx=label2idx
)
ds[10]

{'tokens': '台湾是中国领土不可分割的一部分。完成祖国统一，是大势所趋，民心所向。任何企图制造“两个中国”、“一中一台”、“台湾独立”的图谋，都注定要失败。希望台湾当局以民族大义为重，拿出诚意，采取实际的行动，推动两岸经济文化交流和人员往来，促进两岸直接通邮、通航、通商的早日实现，并尽早回应我们发出的在一个中国的原则下两岸进行谈判的郑重呼吁。',
 'labels': 'BESBEBEBMMESSBESBEBEBESSBMMESBMMESBEBEBESSSBESSSBMMESSSBEBESSBESSBESBESBEBEBESBEBEBESSSBESBEBESBESBEBEBEBEBESBEBESBEBEBEBESBESBESBEBESSBEBEBEBESSBEBESBESBEBEBESBEBES',
 'token_ids': tensor([ 90, 105, 122,  26,  33, 205, 372, 132, 373,  79, 374,   8,  13, 375,
          79,  55, 364, 184, 363,  33, 365,  13,  48, 122, 193, 219, 376, 317,
          48,  39, 136, 376,   3,  55, 377, 378, 225,  21, 169, 379, 167, 168,
         252,  26,  33, 170,  32, 167,  13,  26,  13,  90, 170,  32, 167,  90,
         105, 380, 381, 170,   8,  21, 382,  48, 247, 249, 181, 130, 383, 384,
          55,   6,   7,  90, 105, 385, 272, 111,  39,  95, 193, 150, 386, 129,
          48, 387, 320, 112, 388,  48, 389, 274, 352,  77,   8, 100, 203,  48,
         153, 203, 168, 390, 212, 213, 242, 211, 235, 236,  91,  86, 391, 

In [19]:
def build_dataloader(ds, batch_size, shuffle=False, num_workers=0):
    return DataLoader(
        dataset=ds,
        batch_size=batch_size,
        shuffle=shuffle,
        num_workers=num_workers,
        collate_fn=ds.build_collate_fn()
    )

In [20]:
train_dataloader = build_dataloader(ds, 2)
for batch in train_dataloader:
    print(batch)
    break

{'tokens': ['迈向充满希望的新世纪——一九九八年新年讲话（附图片１张）', '中共中央总书记、国家主席江泽民'], 'labels': ['BEBEBESSBEBEBMMMEBEBESSBESSS', 'BMMEBMESBEBESBE'], 'token_ids': tensor([[ 2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 12, 13, 14, 14, 15, 16,  9,
         16, 17, 18, 19, 20, 21, 22, 23, 24, 25],
        [26, 27, 26, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39,  0,  0,  0,
          0,  0,  0,  0,  0,  0,  0,  0,  0,  0]]), 'label_ids': tensor([[   0,    2,    0,    2,    0,    2,    3,    3,    0,    2,    0,    2,
            0,    1,    1,    1,    2,    0,    2,    0,    2,    3,    3,    0,
            2,    3,    3,    3],
        [   0,    1,    1,    2,    0,    1,    2,    3,    0,    2,    0,    2,
            3,    0,    2, -100, -100, -100, -100, -100, -100, -100, -100, -100,
         -100, -100, -100, -100]])}


#### __2.4 组合训练数据__

基于构造好的数据进行训练方法的构建

In [30]:
def training(model_pkl = "./output/model/last.pkl", total_epoch=2, batch_size=8):
    # 相关参数定义
    train_data_file = "./datas/train.jsonl"
    test_data_file = "./datas/test.jsonl"
    token_json_file = "./datas/tokens.json"
    label_json_files = "./datas/labels.json"

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"当前运行设备为:{device}")

    os.makedirs(os.path.dirname(model_pkl), exist_ok=True)


    # 加载配置信息
    labels = load_json(label_json_files)
    tokenizer = Tokenizer(token_json_file)
    label2idx = {k:v for v, k in enumerate(labels)}
    
    # 数据集加载
    train_ds = TokenClassifyDataset(
        jsonl_file=train_data_file,
        tokenizer=tokenizer, 
        label2idx=label2idx
    )
    train_dataloader = build_dataloader(train_ds, batch_size, shuffle=True)
    test_ds = TokenClassifyDataset(
        jsonl_file=test_data_file,
        tokenizer=tokenizer, 
        label2idx=label2idx
    )
    test_dataloader = build_dataloader(train_ds, batch_size * 2)

    # 构造网络
    network = TokenClassifyNetwork(vocab_size=len(tokenizer.vocabs), num_classes=len(label2idx), hidden_size=128)
    network.to(device=device)
    loss_fn = TokenClassifyLossModule()
    loss_fn.to(device=device)
    opt = optim.SGD(params=network.parameters(), lr=0.01)

    # 遍历数据集进行模型训练
    for epoch in range(total_epoch):
        # 训练
        network.train()
        train_tqdm = tqdm(enumerate(train_dataloader), total=len(train_dataloader))
        for batch_idx, batch in train_tqdm:
            tensor_to_device(batch, device)
            # 前向
            score = network(batch['token_ids'])
            loss = loss_fn(score, batch['label_ids'])

            # 反向
            loss.backward()
            opt.step()
            opt.zero_grad()

            # 日志输出
            train_batch_acc = accuracy(score, batch['label_ids'])
            #print(f"Train Epoch {epoch} Batch {batch_idx} Loss {loss.item():.3f} Accuracy {train_batch_acc.item():.3f}")
            train_tqdm.set_description(f"Train Epoch {epoch} Batch {batch_idx} Loss {loss.item():.3f} Accuracy {train_batch_acc.item():.3f}")

        # 模型评估
        with torch.no_grad():
            network.eval()
            test_tqdm = tqdm(enumerate(test_dataloader), total=len(test_dataloader))
            for batch_idx, batch in test_tqdm:
                tensor_to_device(batch, device)
                # 前向
                score = network(batch['token_ids'])
                loss = loss_fn(score, batch['label_ids'])
    
                # 日志输出
                test_batch_acc = accuracy(score, batch['label_ids'])
                #print(f"Test Epoch {epoch} Batch {batch_idx} Loss {loss.item():.3f} Accuracy {test_batch_acc.item():.3f}")
                test_tqdm.set_description(f"Test Epoch {epoch} Batch {batch_idx} Loss {loss.item():.3f} Accuracy {test_batch_acc.item():.3f}")
        
        # 模型持久化
        save_obj = {
            'epoch': epoch,
            'params': network.state_dict()
        }
        torch.save(save_obj, model_pkl)

In [ ]:
from datetime import datetime

now = datetime.now()

training(
    model_pkl = f"./output/{now.strftime('%Y%m%d_%H%M%S')}/model/last.pkl",
    total_epoch=1000, 
    batch_size=32,
)

当前运行设备为:cuda


Train Epoch 0 Batch 486 Loss 1.337 Accuracy 0.312: 100%|██████████| 487/487 [00:05<00:00, 94.54it/s]
Test Epoch 0 Batch 243 Loss 1.330 Accuracy 0.182: 100%|██████████| 244/244 [00:01<00:00, 125.78it/s]
Train Epoch 1 Batch 486 Loss 1.213 Accuracy 0.353: 100%|██████████| 487/487 [00:05<00:00, 94.60it/s] 
Test Epoch 1 Batch 243 Loss 1.328 Accuracy 0.205: 100%|██████████| 244/244 [00:01<00:00, 125.20it/s]
Train Epoch 2 Batch 486 Loss 1.245 Accuracy 0.332: 100%|██████████| 487/487 [00:05<00:00, 94.69it/s] 
Test Epoch 2 Batch 243 Loss 1.329 Accuracy 0.227: 100%|██████████| 244/244 [00:01<00:00, 125.98it/s]
Train Epoch 3 Batch 486 Loss 1.304 Accuracy 0.294: 100%|██████████| 487/487 [00:05<00:00, 95.24it/s] 
Test Epoch 3 Batch 243 Loss 1.329 Accuracy 0.205: 100%|██████████| 244/244 [00:01<00:00, 125.74it/s]
Train Epoch 4 Batch 486 Loss 1.247 Accuracy 0.212: 100%|██████████| 487/487 [00:05<00:00, 95.12it/s]
Test Epoch 4 Batch 243 Loss 1.329 Accuracy 0.205: 100%|██████████| 244/244 [00:01<00:00,

# 三、模型恢复

In [7]:
model_path = r"./output/20260119_120631/model/last.pkl" 
#model_path = r"./output/20260119_111034/model/last.pkl"

ckpt = torch.load(model_path, map_location='cpu')
print(f"模型epoch: {ckpt['epoch']}")

模型epoch: 999


In [14]:
# 相关参数定义
train_data_file = "./datas/train.jsonl"
test_data_file = "./datas/test.jsonl"
token_json_file = "./datas/tokens.json"
label_json_files = "./datas/labels.json"

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"当前运行设备为:{device}")


# 加载配置信息
labels = load_json(label_json_files)
tokenizer = Tokenizer(token_json_file)
label2idx = {k:v for v, k in enumerate(labels)}

# 构造网络
network = TokenClassifyNetwork(vocab_size=len(tokenizer.vocabs), num_classes=len(label2idx), hidden_size=128)
network.to(device=device)

network

当前运行设备为:cuda


TokenClassifyNetwork(
  (embed_layer): Embedding(2986, 128)
  (rnn_layers): LSTM(128, 128, num_layers=3, batch_first=True)
  (classify): Sequential(
    (0): Linear(in_features=128, out_features=256, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.3, inplace=False)
    (3): Linear(in_features=256, out_features=4, bias=True)
  )
)

In [18]:
# 参数恢复
network.load_state_dict(ckpt['params'])

<All keys matched successfully>

In [42]:
text = "台湾是中国领土不可分割的一部分。完成祖国统一，是大势所趋，民心所向。任何企图制造“两个中国”、“一中一台”、“台湾独立”的图谋，都注定要失败。"
text = "李铁映、贾庆林、曾庆红等领导同志也出席了今晚音乐会。" # SBESSBESSBESBEBESBESBEBMES

tokens = tokenizer.tokenize(text)
token_ids = tokenizer.convert_tokens_to_ids(tokens)

token_ids = torch.tensor([token_ids])
token_ids = token_ids.to(device=device)
score = network(token_ids)

In [44]:
print(score.shape)
pred_ids = torch.argmax(score, dim=-1)
print(pred_ids)

pred_tags = [labels[pred_id] for pred_id in pred_ids[0]]
print(pred_tags)
print("".join(pred_tags))
print(list(zip(text, pred_tags)))

torch.Size([1, 26, 4])
tensor([[3, 0, 2, 3, 3, 0, 2, 3, 3, 0, 2, 3, 0, 2, 0, 2, 3, 0, 2, 3, 0, 2, 0, 2,
         2, 3]], device='cuda:0')
['S', 'B', 'E', 'S', 'S', 'B', 'E', 'S', 'S', 'B', 'E', 'S', 'B', 'E', 'B', 'E', 'S', 'B', 'E', 'S', 'B', 'E', 'B', 'E', 'E', 'S']
SBESSBESSBESBEBESBESBEBEES
[('李', 'S'), ('铁', 'B'), ('映', 'E'), ('、', 'S'), ('贾', 'S'), ('庆', 'B'), ('林', 'E'), ('、', 'S'), ('曾', 'S'), ('庆', 'B'), ('红', 'E'), ('等', 'S'), ('领', 'B'), ('导', 'E'), ('同', 'B'), ('志', 'E'), ('也', 'S'), ('出', 'B'), ('席', 'E'), ('了', 'S'), ('今', 'B'), ('晚', 'E'), ('音', 'B'), ('乐', 'E'), ('会', 'E'), ('。', 'S')]
